# Octo Demo — Octo-Small vs Octo-Base

Hands-on notebook for the Module 4 (VLA Models) presentation. Loads both official Octo
checkpoints and runs inference on an image and a language instruction.

To try it yourself: run every cell top to bottom once (a few minutes on first run), then
re-run the last two cells with your own uploaded photo and instruction.

Runs JAX only — OpenVLA (PyTorch) is kept in a separate notebook (`OpenVLA_Demo.ipynb`)
so the two frameworks don't compete for VRAM on one Colab GPU.

## 1. Mount Google Drive for persistent caching

Colab discards installed packages on every new runtime, but model weights don't need to
be re-downloaded each time if Hugging Face's cache is pointed at Drive instead of the
VM's local disk.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_DIR = '/content/drive/MyDrive/VLA_Demo_Cache'
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = os.path.join(CACHE_DIR, 'huggingface')

# JAX preallocates ~90% of GPU memory by default on import - turn that off so it
# doesn't hog the whole T4 unnecessarily (matters if you ever run anything else
# alongside this in the same runtime).
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'


## 2. Install Octo

Runs JAX on CPU: Octo-Small/Base are small enough (27M / 93M params) that CPU inference
is fast enough for a few demo forward passes, avoiding Colab's GPU CUDA-plugin version
issues entirely. Octo's source and `requirements.txt` are patched below for compatibility
with current library versions, since the repository has not been updated since late
2023.

In [ ]:
# Absolute path + existence check, so re-running this cell without a full
# runtime restart can't land the kernel one directory too deep (that happened:
# a second `%cd octo` from inside an already-`%cd`'d session landed in
# /content/octo/octo, and everything after it failed).
import os
if not os.path.exists('/content/octo'):
    !git clone https://github.com/octo-models/octo.git /content/octo
%cd /content/octo

# Octo's own source (not just requirements.txt) hasn't been updated since JAX 0.4.20
# (Oct 2023) and uses two JAX APIs removed since then: `jax.random.KeyArray`
# (removed in JAX 0.4.24, replace with plain `jax.Array`) and `jax.tree_map`
# (removed in JAX 0.6.0, moved to `jax.tree_util.tree_map`). Both are safe, drop-in
# renames - patch in place before installing.
!sed -i 's/jax\.random\.KeyArray/jax.Array/' octo/utils/typing.py
!grep -rl 'jax\.tree_map(' --include='*.py' . | xargs -r sed -i 's/jax\.tree_map(/jax.tree_util.tree_map(/g'

!pip install -e . -q

# Octo-Small/Base are tiny (27M/93M params) - not worth fighting Colab's GPU plugin
# stack for a handful of demo forward passes, and CPU inference is plenty fast for
# that. Remove any preinstalled CUDA jax plugin so JAX falls back to CPU cleanly:
# Colab ships jax-cuda13-plugin pre-installed, and installing a jax[cuda12] extra on
# top of it previously left two CUDA plugins fighting to register the same device
# ("ALREADY_EXISTS: PJRT_Api already exists for device type cuda").
!pip uninstall -y -q jax-cuda12-plugin jax-cuda12-pjrt jax-cuda13-plugin jax-cuda13-pjrt 2>/dev/null

# Install jax (CPU-only, no cuda extra) together with flax/chex/optax/distrax in ONE
# resolver pass so they land on a mutually compatible set. Installing jax and flax in
# separate pip calls previously left flax newer than jax, causing
# "AttributeError: module 'jax.core' has no attribute 'get_opaque_trace_state'".
!pip install -U -q jax flax chex optax distrax

# Pinning numpy to the EXACT version verified working (2.2.6), not just a range:
# numpy>=1.26,<2.3 still let pip pick different point releases with their own
# bugs - 2.5.4 crashed with a missing _blas_supports_fpe attribute, and a
# different release in the <2.3 range crashed in numpy.strings's own
# _override___module__() trying to set .__module__ on a ufunc object. 2.2.6 is
# the one actually confirmed to load and run Octo correctly end-to-end.
# Octo's requirements.txt hard-pins exact late-2023 versions (numpy==1.24.3,
# tensorflow==2.15.0, ...) which have no prebuilt wheels for Colab's current Python
# (3.13) and fail to build from source (the same kind of dependency rot hit in
# OpenVLA_Demo.ipynb, but surfacing as a build failure here instead of a silent bug).
# Installing the same packages WITHOUT the exact pins lets pip pick modern,
# 3.13-compatible versions instead.
!pip install -q \
    "numpy==2.2.6" \
    ml_dtypes \
    tensorflow tensorflow_probability tensorflow_hub tensorflow_text \
    tensorflow_datasets tensorflow_graphics \
    gym ml_collections tqdm absl-py scipy wandb einops imageio moviepy \
    "transformers>=4.34.1,<5.0" plotly matplotlib

# dlimp's own setup.py separately hard-pins tensorflow==2.15.0, which conflicts with
# the modern tensorflow installed above (another dead pin, no 3.13 wheel). --no-deps
# skips dlimp's pin - it's a thin tf.data wrapper, it just needs *a* working
# tensorflow, which we already have.
!pip install -q --no-deps \
    "dlimp @ git+https://github.com/kvablack/dlimp@5edaa4691567873d495633f2708982b42edf1972"

# Octo's text tokenizer uses `transformers.FlaxT5EncoderModel` to encode language
# instructions. transformers v5 dropped Flax/TensorFlow support entirely (PyTorch
# only), which deletes that class - same kind of break as OpenVLA's
# `AutoModelForVision2Seq` removal. Flax classes are kept through the whole v4.x
# line (maintained as LTS), hence the <5.0 cap above.


## 2b. Patch stale-dependency bugs (in Octo's dependencies, not Octo itself)

`tensorflow_probability`'s JAX backend (via `distrax`, used by the diffusion action head)
and `transformers`' legacy Flax T5 code (used by the language tokenizer) reference JAX
APIs removed in current JAX, with no newer release fixing either. Three narrow patches
are applied to the installed files below, verified locally before use here: a guard
around a `tf.Variable`-emulation registration Octo never exercises, a JAX keyword
argument rename (`a_max=` → `max=`), and a fix for a NumPy ufunc-attribute crash specific
to Python 3.13.

In [ ]:
import tensorflow_probability, transformers, os

# tensorflow_probability's JAX backend registers a type handler using JAX internals
# removed in current JAX (jax.interpreters.xla.pytype_aval_mappings / 
# canonicalize_dtype_handlers). tensorflow_probability and distrax are both already
# at their latest available PyPI release - no newer version fixes this. The
# registration is only for emulating mutable tf.Variable semantics under JAX, never
# exercised by Octo's actual usage, so it's safe to skip if the attributes are gone.
tfp_ops_path = os.path.join(os.path.dirname(tensorflow_probability.__file__),
                             "python", "internal", "backend", "jax", "ops.py")
with open(tfp_ops_path) as f:
    _content = f.read()
_old = """if JAX_MODE:
  jax.interpreters.xla.canonicalize_dtype_handlers[NumpyVariable] = (
      jax.interpreters.xla.canonicalize_dtype_handlers[onp.ndarray])
  jax.interpreters.xla.pytype_aval_mappings[NumpyVariable] = (
      jax.interpreters.xla.pytype_aval_mappings[onp.ndarray])
  jax.core.pytype_aval_mappings[NumpyVariable] = (
      jax.core.pytype_aval_mappings[onp.ndarray])"""
_new = """if JAX_MODE:
  try:
    jax.interpreters.xla.canonicalize_dtype_handlers[NumpyVariable] = (
        jax.interpreters.xla.canonicalize_dtype_handlers[onp.ndarray])
    jax.interpreters.xla.pytype_aval_mappings[NumpyVariable] = (
        jax.interpreters.xla.pytype_aval_mappings[onp.ndarray])
  except AttributeError:
    pass
  try:
    jax.core.pytype_aval_mappings[NumpyVariable] = (
        jax.core.pytype_aval_mappings[onp.ndarray])
  except AttributeError:
    pass"""
if _old in _content:
    with open(tfp_ops_path, "w") as f:
        f.write(_content.replace(_old, _new))
    print("Patched tensorflow_probability ops.py")
else:
    print("tfp ops.py already patched or upstream content changed - check manually if loading still fails")

# transformers' legacy FlaxT5 code (used by Octo's language-instruction tokenizer
# via FlaxT5EncoderModel) calls jnp.clip(..., a_max=...) - JAX renamed that kwarg to
# `max=`. transformers' Flax classes are deprecated/unmaintained, hence the drift.
t5_flax_path = os.path.join(os.path.dirname(transformers.__file__),
                             "models", "t5", "modeling_flax_t5.py")
with open(t5_flax_path) as f:
    _content = f.read()
if "a_max=" in _content:
    with open(t5_flax_path, "w") as f:
        f.write(_content.replace("a_max=", "max="))
    print("Patched transformers modeling_flax_t5.py")
else:
    print("transformers flax_t5.py already patched or upstream content changed - check manually if loading still fails")


# numpy's own `numpy/_core/strings.py` (imported transitively via
# tensorflow -> scipy.sparse -> scipy's array_api_compat -> `from numpy import *`)
# crashes on import on Colab's Python 3.13: its `_override___module__()` tries to
# set `.__module__` on ufunc objects that don't support attribute assignment under
# this numpy build/Python version combo (ufuncs only gained a writable __dict__ in
# a later numpy release; this is purely cosmetic metadata, safe to skip).
import numpy, os as _os
_np_strings_path = _os.path.join(_os.path.dirname(numpy.__file__), "_core", "strings.py")
with open(_np_strings_path) as f:
    _content = f.read()
_old = """def _override___module__():
    for ufunc in [
        isalnum, isalpha, isdecimal, isdigit, islower, isnumeric, isspace,
        istitle, isupper, str_len,
    ]:
        ufunc.__module__ = "numpy.strings"
        ufunc.__qualname__ = ufunc.__name__"""
_new = """def _override___module__():
    for ufunc in [
        isalnum, isalpha, isdecimal, isdigit, islower, isnumeric, isspace,
        istitle, isupper, str_len,
    ]:
        try:
            ufunc.__module__ = "numpy.strings"
            ufunc.__qualname__ = ufunc.__name__
        except AttributeError:
            pass"""
if _old in _content:
    with open(_np_strings_path, "w") as f:
        f.write(_content.replace(_old, _new))
    print("Patched numpy _core/strings.py")
else:
    print("numpy strings.py already patched or upstream content changed - check manually if loading still fails")


## 3. Load Octo-Small and Octo-Base

Both checkpoints are small enough to load into the same JAX runtime together without a
VRAM conflict.

In [ ]:
from octo.model.octo_model import OctoModel
import jax

octo_small = OctoModel.load_pretrained("hf://rail-berkeley/octo-small-1.5")
print("Octo-Small loaded:")
print(octo_small.get_pretty_spec())


Loading the second checkpoint, used for the Octo-Small vs Octo-Base comparison
in Section 6.

In [ ]:
octo_base = OctoModel.load_pretrained("hf://rail-berkeley/octo-base-1.5")
print("Octo-Base loaded:")
print(octo_base.get_pretty_spec())


## 4. Helper functions: run inference + visualize

`run_octo` follows the single-step inference pattern from Octo's own
`examples/01_inference_pretrained.ipynb`: resize to 256×256, add batch and time-history
dimensions, build a language task, and unnormalize the output using one of the
pretraining mixture's dataset statistics.

`unnorm_key="bridge_dataset"` matches the WidowX / BridgeData-V2-style setup used in
Octo's official example. For a very different scene type, run
`print(list(octo_small.dataset_statistics.keys()))` and choose a closer match — this
affects the absolute action values, not the relative comparison between Octo-Small and
Octo-Base on the same input.

`visualize_action` draws an arrow for the predicted step, for illustration only:
- Octo's output is a single-step, low-level 7-DoF control delta in the robot's own
  frame, not a detected pixel location or grasp target.
- The arrow's start point is the image's geometric center, a placeholder since a static
  photo gives no real gripper position; its length is scaled up (`ARROW_SCALE`) because
  the real deltas are only a few millimeters.
- Octo is meant to run as a closed-loop controller, re-evaluated on an updated camera
  frame every ~100–200 ms. A single prediction from one static photo is only the first
  step of that loop, so the arrow's position relative to objects in the photo should not
  be read as confirmation of which object the model identified as the target.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

ACTION_DIM_LABELS = ['x', 'y', 'z', 'yaw', 'pitch', 'roll', 'grasp']

def run_octo(model, image_rgb_256, instruction, unnorm_key="bridge_dataset"):
    """image_rgb_256: HxWx3 uint8 array already resized to 256x256.
    Returns the first predicted action of the chunk, shape (7,)."""
    img = image_rgb_256[np.newaxis, np.newaxis, ...]
    observation = {"image_primary": img, "timestep_pad_mask": np.array([[True]])}
    task = model.create_tasks(texts=[instruction])
    action = model.sample_actions(
        observation, task,
        unnormalization_statistics=model.dataset_statistics[unnorm_key]["action"],
        rng=jax.random.PRNGKey(0),
    )
    return np.array(action)[0, 0]  # drop batch dim, take first step of the predicted chunk

def visualize_action(image_rgb_256, action, title=""):
    """Draws the predicted x/y step as an arrow from the image's geometric CENTER
    (not a detected gripper position - see the markdown note above for why), with the
    arrow length exaggerated (ARROW_SCALE) purely so a few-millimeter step is visible
    at all. Do not read the arrow as "this is the object the model is grasping" - see
    the markdown note above."""
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(image_rgb_256)
    h, w = image_rgb_256.shape[:2]
    cx, cy = w / 2, h / 2
    ARROW_SCALE = 1500  # raw deltas are ~0.001-0.03; *100 was visually indistinguishable from zero
    dx, dy = float(action[0]) * ARROW_SCALE, -float(action[1]) * ARROW_SCALE
    max_len = 0.4 * min(w, h)
    length = (dx ** 2 + dy ** 2) ** 0.5
    if length > max_len and length > 0:
        dx, dy = dx * max_len / length, dy * max_len / length
    ax.plot(cx, cy, 'o', color='yellow', markersize=6, markeredgecolor='black')
    ax.arrow(cx, cy, dx, dy, head_width=8, color='red', length_includes_head=True)
    ax.set_title(title)
    ax.axis('off')
    plt.show()
    print({k: round(float(v), 3) for k, v in zip(ACTION_DIM_LABELS, action)})


## 5. Try it yourself — upload a photo and type an instruction

Upload any photo, type an instruction when prompted, and re-run the cell to see both
models react.

In [ ]:
from google.colab import files

print("Upload an image (jpg/png):")
uploaded = files.upload()
IMAGE_PATH = list(uploaded.keys())[0]

INSTRUCTION = input("Type an instruction (e.g. 'pick up the spatula'): ").strip()
if not INSTRUCTION:
    INSTRUCTION = "pick up the object"

img = np.array(Image.open(IMAGE_PATH).convert("RGB").resize((256, 256)))
plt.imshow(img); plt.axis('off'); plt.title("Input image"); plt.show()

action_small = run_octo(octo_small, img, INSTRUCTION)
visualize_action(img, action_small, title="Octo-Small prediction")

action_base = run_octo(octo_base, img, INSTRUCTION)
visualize_action(img, action_base, title="Octo-Base prediction")


## 6. Pre-loaded comparison pairs (for the slide deck)

Fixed set of (image, instruction) pairs run through both models, to check the Octo
paper's own finding — Octo-Base is more robust to scene variation and less prone to
premature grasp attempts than Octo-Small — against these test cases.

Pair 3 ("pick up the fruit") should be read together with the note on `visualize_action`
in Section 4: the arrow's apparent position relative to the pineapple-shaped toy and the
pot is not reliable evidence of which object either model targeted, since a single
static-image prediction can't confirm correct semantic grounding on its own. Worth
presenting as a discussion point on the limits of single-step VLA demos rather than a
pass/fail result.

In [ ]:
TEST_PAIRS = [
    # All 4 images are real BridgeData V2 frames (same toy-kitchen environment
    # Octo's own official example uses) - see test_images/README.md for what's in each.
    ("/content/drive/MyDrive/VLA_Demo_Cache/test_images/pair1_cup.jpg",
     "pick up the spatula"),  # baseline sanity check
    ("/content/drive/MyDrive/VLA_Demo_Cache/test_images/pair2_fork_cup.jpg",
     "pick up the spoon, not the pot"),  # distractor / language grounding
    ("/content/drive/MyDrive/VLA_Demo_Cache/test_images/pair3_fruit.jpg",
     "pick up the fruit"),  # semantic/novel-instruction (RT-2-style)
    ("/content/drive/MyDrive/VLA_Demo_Cache/test_images/pair4_cluttered.jpg",
     "pick up the spatula"),  # scene-robustness (Octo-Small vs Octo-Base)
]

for image_path, instruction in TEST_PAIRS:
    img = np.array(Image.open(image_path).convert("RGB").resize((256, 256)))
    print(f"=== Instruction: {instruction!r} ===")
    visualize_action(img, run_octo(octo_small, img, instruction), title="Octo-Small")
    visualize_action(img, run_octo(octo_base, img, instruction), title="Octo-Base")
